# Rigorous Forecast Evaluation

**Goal:** Evaluate nowcast performance using proper scoring rules, statistical tests, and out-of-sample backtesting.

**What you'll learn:**
- Implement expanding-window backtesting framework
- Compute multiple evaluation metrics (MSE, MAE, CRPS, directional accuracy)
- Perform Diebold-Mariano test for forecast comparison
- Visualize forecast errors across business cycles
- Compare DFM to benchmarks (AR, random walk)

**Time:** 12-15 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from statsmodels.tsa.ar_model import AutoReg
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Forecast Evaluation Toolkit Loaded")

## Quick Win: Out-of-Sample Backtest in 3 Minutes

In [ ]:
# Generate data (same as previous notebook)
np.random.seed(42)
dates = pd.date_range('2000-01-01', '2024-03-01', freq='M')
T = len(dates)

factor = np.zeros(T)
for t in range(1, T):
    factor[t] = 0.7 * factor[t-1] + np.random.randn()

monthly_data = pd.DataFrame({
    'IP': 0.8 * factor + 0.5 * np.random.randn(T),
    'Employment': 0.7 * factor + 0.6 * np.random.randn(T),
    'Retail_Sales': 0.6 * factor + 0.7 * np.random.randn(T),
    'PMI': 0.5 * factor + 0.8 * np.random.randn(T)
}, index=dates)

monthly_data = (monthly_data - monthly_data.mean()) / monthly_data.std()

quarterly_factor = pd.Series(factor, index=dates).resample('Q').mean()
gdp = 2.0 + 1.5 * quarterly_factor + 0.3 * np.random.randn(len(quarterly_factor))
gdp.name = 'GDP_Growth'

print(f"Data loaded: {len(monthly_data)} months, {len(gdp)} quarters")

In [ ]:
def expanding_window_nowcast(monthly_data, gdp, start_date='2010-01-01', n_factors=3):
    """
    Pseudo-real-time backtesting: expand training window at each forecast.
    """
    results = []
    
    # Quarterly forecast dates
    forecast_dates = pd.date_range(start_date, gdp.index[-1], freq='Q')
    
    for q_date in forecast_dates:
        # Training data: everything up to (but not including) forecast quarter
        train_months = monthly_data.loc[:q_date].iloc[:-3]  # Exclude current quarter
        train_gdp = gdp.loc[:q_date].iloc[:-1]
        
        if len(train_gdp) < 20:  # Need minimum training data
            continue
        
        try:
            # Estimate DFM on training data
            dfm = DynamicFactor(train_months, k_factors=n_factors, factor_order=1)
            dfm_res = dfm.fit(maxiter=500, disp=False)
            
            # Extract factors and aggregate to quarterly
            factors_m = dfm_res.factors.filtered
            factors_q = factors_m.resample('Q').mean()
            
            # Bridge equation
            common_idx = factors_q.index.intersection(train_gdp.index)
            X_bridge = factors_q.loc[common_idx]
            y_bridge = train_gdp.loc[common_idx]
            
            bridge = LinearRegression()
            bridge.fit(X_bridge, y_bridge)
            
            # Nowcast current quarter (use all available monthly data)
            current_factors = monthly_data.loc[:q_date].iloc[-3:]
            if len(current_factors) < 3:
                continue
                
            # Re-extract factors including current quarter
            dfm_current = DynamicFactor(
                monthly_data.loc[:q_date], 
                k_factors=n_factors, 
                factor_order=1
            )
            dfm_current_res = dfm_current.fit(maxiter=500, disp=False)
            factors_current = dfm_current_res.factors.filtered.iloc[-3:].mean().values.reshape(1, -1)
            
            nowcast = bridge.predict(factors_current)[0]
            
            # Actual GDP (first release)
            actual = gdp.loc[q_date]
            
            results.append({
                'date': q_date,
                'forecast': nowcast,
                'actual': actual,
                'error': actual - nowcast
            })
        except:
            continue
    
    return pd.DataFrame(results)

# Run backtest
print("Running expanding-window backtest...")
results_dfm = expanding_window_nowcast(monthly_data, gdp, start_date='2010-01-01')
print(f"✓ Generated {len(results_dfm)} out-of-sample forecasts")
print(f"\nFirst few results:")
print(results_dfm.head())

## Benchmark Models: AR(1) and Random Walk

In [ ]:
def ar1_backtest(gdp, start_date='2010-01-01'):
    """AR(1) benchmark: GDP_t = α + β·GDP_{t-1} + ε"""
    results = []
    forecast_dates = pd.date_range(start_date, gdp.index[-1], freq='Q')
    
    for q_date in forecast_dates:
        train = gdp.loc[:q_date].iloc[:-1]
        if len(train) < 10:
            continue
        
        # Fit AR(1)
        model = AutoReg(train, lags=1)
        model_fit = model.fit()
        
        # Forecast
        forecast = model_fit.predict(start=len(train), end=len(train))[0]
        actual = gdp.loc[q_date]
        
        results.append({
            'date': q_date,
            'forecast': forecast,
            'actual': actual,
            'error': actual - forecast
        })
    
    return pd.DataFrame(results)

def rw_backtest(gdp, start_date='2010-01-01'):
    """Random walk benchmark: forecast = last observation"""
    results = []
    forecast_dates = pd.date_range(start_date, gdp.index[-1], freq='Q')
    
    for q_date in forecast_dates:
        prev_gdp = gdp.loc[:q_date].iloc[-2] if len(gdp.loc[:q_date]) >= 2 else gdp.mean()
        actual = gdp.loc[q_date]
        
        results.append({
            'date': q_date,
            'forecast': prev_gdp,
            'actual': actual,
            'error': actual - prev_gdp
        })
    
    return pd.DataFrame(results)

print("Running benchmark models...")
results_ar = ar1_backtest(gdp)
results_rw = rw_backtest(gdp)
print(f"✓ AR(1): {len(results_ar)} forecasts")
print(f"✓ Random Walk: {len(results_rw)} forecasts")

## Evaluation Metrics: The Scorecard

In [ ]:
def compute_metrics(results_df):
    """Compute comprehensive evaluation metrics."""
    errors = results_df['error'].values
    forecasts = results_df['forecast'].values
    actuals = results_df['actual'].values
    
    metrics = {
        'MSE': np.mean(errors**2),
        'RMSE': np.sqrt(np.mean(errors**2)),
        'MAE': np.mean(np.abs(errors)),
        'Bias': np.mean(errors),
        'Directional_Accuracy': np.mean(np.sign(forecasts) == np.sign(actuals)) * 100,
        'N_forecasts': len(errors)
    }
    
    return metrics

# Compute for all models
metrics_dfm = compute_metrics(results_dfm)
metrics_ar = compute_metrics(results_ar)
metrics_rw = compute_metrics(results_rw)

# Display comparison table
comparison = pd.DataFrame({
    'DFM Nowcast': metrics_dfm,
    'AR(1) Benchmark': metrics_ar,
    'Random Walk': metrics_rw
})

print("\n" + "="*60)
print("FORECAST EVALUATION SCORECARD")
print("="*60)
print(comparison.round(3))
print("="*60)

# Determine winner
best_rmse = comparison.loc['RMSE'].idxmin()
print(f"\n🏆 Best RMSE: {best_rmse} ({comparison.loc['RMSE', best_rmse]:.3f})")

improvement = (metrics_ar['RMSE'] - metrics_dfm['RMSE']) / metrics_ar['RMSE'] * 100
print(f"DFM improves over AR(1) by {improvement:.1f}%")

## Diebold-Mariano Test: Statistical Significance

In [ ]:
def diebold_mariano_test(errors_a, errors_b, h=1, loss_fn=lambda e: e**2):
    """
    Test whether forecast A is significantly better than forecast B.
    
    H0: Equal predictive accuracy
    Ha: Forecast A has lower expected loss
    """
    # Loss differential
    d = loss_fn(errors_a) - loss_fn(errors_b)
    T = len(d)
    
    # Mean differential
    d_bar = np.mean(d)
    
    # Variance (simplified: assumes no autocorrelation)
    var_d = np.var(d, ddof=1) / T
    
    # DM statistic
    dm_stat = d_bar / np.sqrt(var_d)
    
    # Harvey small-sample correction
    dm_stat_corrected = dm_stat * np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    
    # P-value (two-sided)
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat_corrected)))
    
    return dm_stat_corrected, p_value

# Align results (use common dates)
common_dates = results_dfm['date'].values
errors_dfm = results_dfm['error'].values
errors_ar = results_ar[results_ar['date'].isin(common_dates)]['error'].values

# Conduct test: DFM vs AR(1)
dm_stat, p_val = diebold_mariano_test(errors_dfm, errors_ar, h=1)

print("\nDiebold-Mariano Test: DFM vs AR(1)")
print("="*50)
print(f"DM Statistic: {dm_stat:.3f}")
print(f"P-value: {p_val:.4f}")
print("\nInterpretation:")
if p_val < 0.05:
    winner = "DFM" if dm_stat < 0 else "AR(1)"
    print(f"  ✓ {winner} is SIGNIFICANTLY better (p < 0.05)")
    if dm_stat < 0:
        print(f"  → DFM has lower forecast error than AR(1) benchmark")
else:
    print(f"  • No significant difference (p = {p_val:.3f})")
    print(f"  → Both models have similar forecast accuracy")

## Visualize Forecast Errors Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel A: Actual vs Forecasts
axes[0].plot(results_dfm['date'], results_dfm['actual'], 'o-', 
             label='Actual GDP', linewidth=2, markersize=5, color='black')
axes[0].plot(results_dfm['date'], results_dfm['forecast'], 's--', 
             label='DFM Nowcast', linewidth=1.5, markersize=4, alpha=0.7)
axes[0].plot(results_ar['date'], results_ar['forecast'], '^:', 
             label='AR(1)', linewidth=1.5, markersize=4, alpha=0.7)

axes[0].axhline(0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].set_ylabel('GDP Growth (%)', fontsize=11)
axes[0].set_title('Out-of-Sample Forecasts vs Actual', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].grid(alpha=0.3)

# Panel B: Forecast Errors
axes[1].bar(results_dfm['date'], results_dfm['error'], 
            width=80, label='DFM Error', alpha=0.6, color='C0')
axes[1].bar(results_ar['date'], results_ar['error'], 
            width=40, label='AR(1) Error', alpha=0.6, color='C1')

axes[1].axhline(0, color='k', linestyle='-', linewidth=0.8)
axes[1].set_ylabel('Forecast Error (pp)', fontsize=11)
axes[1].set_xlabel('Quarter', fontsize=11)
axes[1].set_title('Forecast Errors Over Time', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper left', fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Identify worst misses
worst_errors = results_dfm.nlargest(3, 'error', keep='all')[['date', 'forecast', 'actual', 'error']]
print("\nWorst Nowcast Misses (largest positive errors):")
print(worst_errors)

## Modify This: Sensitivity Analysis

In [ ]:
# CUSTOMIZE THESE
N_FACTORS_LIST = [1, 2, 3, 4, 5]  # Try different factor numbers

print("Sensitivity Analysis: How many factors to use?\n")
sensitivity_results = []

for n_fac in N_FACTORS_LIST:
    results = expanding_window_nowcast(monthly_data, gdp, start_date='2010-01-01', n_factors=n_fac)
    metrics = compute_metrics(results)
    
    sensitivity_results.append({
        'N_Factors': n_fac,
        'RMSE': metrics['RMSE'],
        'MAE': metrics['MAE'],
        'Dir_Acc': metrics['Directional_Accuracy']
    })
    
    print(f"  {n_fac} factors: RMSE={metrics['RMSE']:.3f}, Dir. Acc={metrics['Directional_Accuracy']:.1f}%")

sensitivity_df = pd.DataFrame(sensitivity_results)

# Plot sensitivity
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sensitivity_df['N_Factors'], sensitivity_df['RMSE'], 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Factors', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Forecast Accuracy vs Number of Factors', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_n = sensitivity_df.loc[sensitivity_df['RMSE'].idxmin(), 'N_Factors']
print(f"\n→ Optimal number of factors: {int(best_n)} (minimizes RMSE)")

## Go Deeper

**Extensions:**
1. **State-dependent evaluation:** Compute metrics separately for recessions vs expansions
2. **Forecast combination:** Optimally combine DFM + AR using inverse-variance weights
3. **CRPS evaluation:** Compute for full predictive distributions (requires uncertainty quantification)
4. **Rolling window:** Use fixed window (e.g., last 10 years) instead of expanding

**Real-world comparison:**
- Download NY Fed Nowcast historical estimates
- Compare your DFM to their operational model
- Analyze when/why they diverge

**Production monitoring:**
- Track forecast errors in real-time dashboard
- Alert when error exceeds 2× historical std
- Automatically retrain if performance degrades